# Surgical Instrument Force & Motion Analysis

Analyzes sequence recordings (`*.igs.mha`, IGSIO/PLUS metafile format) of surgical
training trials, organized **one subfolder per participant**:

```
data/
  P01/  trial1.igs.mha  trial2.igs.mha  trial3.igs.mha
  P02/  trial1.igs.mha  trial2.igs.mha  trial3.igs.mha
  ...
```

Each file contains, per frame:

| Field | Meaning |
|---|---|
| `*TipToWorldTransform` | 4×4 pose of each instrument tip (**Bipolar**, **Cavitron**, **Scissors**) |
| `Force` | `fx fy fz tx ty tz` — 3D force + 3D torque from the force sensor |
| `BipolarCollectedPoint0..3` | 4 fiducial points used to register trials into a common frame |
| `Timestamp` | frame time in seconds |

**What this notebook produces**

1. **Point-wise rigid registration** of every trial onto a single common reference
   frame, using the 4 collected fiducials (all trials, all participants share it).
2. **Time normalized to [0, 1]** per trial so recordings of different length align.
3. **Force magnitude** `√(fx²+fy²+fz²)` — one panel per participant.
4. **Velocity, acceleration, jerk** per instrument per trial — one figure per participant.
5. **3D trajectories** as time-colored lines — one figure per participant (trials × instruments).
6. **Summative figures**: per-participant averages and a cross-participant comparison
   of average force / velocity / acceleration / jerk.

> Runs as-is in **Google Colab**. Upload your participant folders (or a zip) when
> prompted, or mount Google Drive and point `DATA_DIR` at the folder that holds them.

## 1 · Setup

In [ ]:
# Colab already ships numpy / scipy / matplotlib; this is a no-op there and a
# convenience when running elsewhere.
import importlib, subprocess, sys
for pkg in ("numpy", "scipy", "matplotlib", "pandas"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, re, glob, warnings
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection

# In-use masking leaves all-NaN slices for instruments never in use in a trial;
# nanmean/nanmax on those are expected and return NaN — silence the routine warnings.
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="All-NaN slice encountered")

# ---- Plot style ---------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 10, "legend.fontsize": 9, "legend.frameon": False,
    "font.size": 10,
})

INSTRUMENTS = ["Bipolar", "Cavitron", "Scissors"]
# Okabe-Ito colorblind-safe palette, fixed order per instrument.
INST_COLOR = {"Bipolar": "#0072B2", "Cavitron": "#E69F00", "Scissors": "#009E73"}
# Sequential map used everywhere "value = time".
TIME_CMAP = "viridis"
print("Setup complete.")

## 2 · Provide the data as a single zip

Package the whole study as **one `.zip`** whose contents are the participant
subfolders, each holding that participant's trial `.igs.mha` files:

```
study.zip
 └─ P01/  trial1.igs.mha  trial2.igs.mha  trial3.igs.mha
 └─ P02/  trial1.igs.mha  trial2.igs.mha  trial3.igs.mha
 └─ ...
```

Set `DATA_ZIP` to that file. The zip is extracted and the participant / trial
structure is recovered automatically.

- An extra wrapper folder inside the zip (e.g. everything under a top-level `data/`)
  is fine — participants are detected by the folder that directly contains the
  `.igs.mha` files, at any depth.
- **Colab** — run the upload cell to pick the zip, or mount Google Drive and point
  `DATA_ZIP` at it.

In [ ]:
DATA_ZIP = "data/study.zip"   # the single zip containing the participant subfolders
WORK_DIR = "data_unzipped"    # where the zip is extracted (recreated on each run)
TIME_UNIT = "s"               # timestamps are in seconds
SMOOTH_WINDOW = 11            # Savitzky-Golay window (odd, in frames) for differentiation; 0 disables

In [ ]:
# --- Optional: upload the zip directly in Colab ---------------------------
# Uncomment in Colab, then pick your study .zip. It is saved to DATA_ZIP.
#
# from google.colab import files
# os.makedirs(os.path.dirname(DATA_ZIP) or ".", exist_ok=True)
# up = files.upload()
# name = next(n for n in up if n.lower().endswith(".zip"))
# with open(DATA_ZIP, "wb") as f:
#     f.write(up[name])
# print("saved", name, "->", DATA_ZIP)
#
# --- Optional: mount Google Drive instead ---------------------------------
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_ZIP = "/content/drive/MyDrive/study.zip"

In [ ]:
import zipfile, shutil

assert os.path.isfile(DATA_ZIP), (
    f"Zip not found: {DATA_ZIP!r}. Upload it (see the cell above) or fix DATA_ZIP.")

# Fresh extraction every run so stale files never linger.
if os.path.isdir(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as z:
    z.extractall(WORK_DIR)
print(f"Extracted {DATA_ZIP} -> {WORK_DIR}/")


def discover_participants(root):
    "Recursively find trials; participant id = folder that directly holds the .igs.mha files."
    parts = {}
    for f in sorted(glob.glob(os.path.join(root, "**", "*.igs.mha"), recursive=True)):
        if "__MACOSX" in f:                       # skip macOS zip cruft
            continue
        pid = os.path.basename(os.path.dirname(f))
        parts.setdefault(pid, []).append(f)
    return {pid: sorted(fs) for pid, fs in parts.items()}

participant_files = discover_participants(WORK_DIR)
assert participant_files, (
    f"No .igs.mha files inside {DATA_ZIP!r}. Expected participant subfolders "
    "like P01/trial1.igs.mha.")

print(f"\nFound {len(participant_files)} participant(s):")
for pid, fs in participant_files.items():
    print(f"  {pid}: {len(fs)} trial(s)")
    for f in fs:
        print(f"      {os.path.basename(f)}")

## 3 · Parse the sequence files (and flag missing tracking)

The `.igs.mha` header is a flat list of `Key = Value` lines; per-frame fields are
named `Seq_Frame<idx>_<Field>`. We read tracking, force, timestamps and the fiducial
points — the referenced video is ignored. Every trial is tagged with its participant.

**Preprocessing — missing-tracking detection.** When an instrument is not tracked in a
frame, its `*TipToWorldTransform` 4×4 matrix is held **exactly** equal to the previous
frame's. We flag every such frame and build a per-instrument **`tracking_status`**
array: `0` = missing (frozen pose), `1` = tracked. Frame 0 has no predecessor and is
treated as tracked.

In [ ]:
def parse_mha(path, participant, trial_no):
    header, frames = {}, {}
    with open(path, "r", errors="ignore") as fh:
        for line in fh:
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip()
            m = re.match(r"Seq_Frame(\d+)_(.+)", k)
            if m:
                frames.setdefault(int(m.group(1)), {})[m.group(2)] = v
            else:
                header[k] = v
            if k == "ElementDataFile":   # end of header; pixel data (if any) follows
                break

    idx = sorted(frames)

    def as_mat(fr, key):
        return np.array([float(x) for x in fr[key].split()], float).reshape(4, 4)

    n_pts = int(header.get("BipolarCollectedPointCount", 0))
    fiducials = (np.array([[float(x) for x in header[f"BipolarCollectedPoint{i}"].split()]
                           for i in range(n_pts)]) if n_pts else None)

    timestamps = np.array([float(frames[i]["Timestamp"]) for i in idx])
    force = np.array([[float(x) for x in frames[i]["Force"].split()] for i in idx])  # fx fy fz tx ty tz

    poses = {}
    for inst in INSTRUMENTS:
        key = inst + "TipToWorldTransform"
        if key in frames[idx[0]]:
            poses[inst] = np.stack([as_mat(frames[i], key) for i in idx])            # (N,4,4)

    # Preprocessing: an untracked instrument keeps the exact same 4x4 pose as the
    # previous frame. Flag those frames as missing -> tracking_status 0 (missing) / 1 (tracked).
    tracking_status = {}
    for inst, M in poses.items():
        st = np.ones(len(M), dtype=int)              # frame 0 has no predecessor -> tracked
        frozen = np.all(M[1:] == M[:-1], axis=(1, 2))   # pose identical to previous frame
        st[1:][frozen] = 0
        tracking_status[inst] = st

    return dict(participant=participant, trial=trial_no,
                name=os.path.basename(path).replace(".igs.mha", ""),
                label=f"{participant}·T{trial_no}",
                header=header, timestamps=timestamps, force=force,
                poses=poses, tracking_status=tracking_status, fiducials=fiducials)


trials = []                       # flat list of all trials, in participant/trial order
participants = {}                 # participant_id -> list of its trials
for pid, fs in participant_files.items():
    participants[pid] = []
    for t_no, f in enumerate(fs, start=1):
        tr = parse_mha(f, pid, t_no)
        trials.append(tr)
        participants[pid].append(tr)

for tr in trials:
    dur = tr["timestamps"][-1] - tr["timestamps"][0]
    tracked = "  ".join(f"{inst}:{100 * tr['tracking_status'][inst].mean():5.1f}%"
                        for inst in tr["poses"])
    print(f"{tr['label']:<12s} {len(tr['timestamps']):5d} frames {dur:6.1f} {TIME_UNIT}  "
          f"tracked -> {tracked}")

## 4 · Point-wise registration to a common reference frame

Every trial carries the **same 4 physical fiducials** (`BipolarCollectedPoint0..3`).
We compute the rigid transform (rotation + translation, no scaling) that best maps
each trial's fiducials onto a single **reference set** — by default the first trial
of the first participant — using the closed-form SVD solution (Kabsch / Horn's
absolute orientation). Applying that transform to every `*TipToWorldTransform`
position places **all trials of all participants** in one shared frame, so
trajectories are directly comparable.

The per-trial fiducial **RMSE** flags a mislabeled or mis-collected point.

In [ ]:
def rigid_register(src, dst):
    "Least-squares rigid transform (4x4) mapping src points onto dst; returns (G, rmse)."
    src, dst = np.asarray(src, float), np.asarray(dst, float)
    cs, cd = src.mean(0), dst.mean(0)
    H = (src - cs).T @ (dst - cd)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))          # guard against reflection
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    t = cd - R @ cs
    G = np.eye(4); G[:3, :3] = R; G[:3, 3] = t
    rmse = np.sqrt(np.mean(np.sum(((R @ src.T).T + t - dst) ** 2, axis=1)))
    return G, rmse


ref_pts = trials[0]["fiducials"]                    # common reference for every trial

for tr in trials:
    if tr["fiducials"] is not None and ref_pts is not None:
        tr["G"], tr["rmse"] = rigid_register(tr["fiducials"], ref_pts)
    else:
        tr["G"], tr["rmse"] = np.eye(4), np.nan

    tr["pos"] = {}
    for inst, P in tr["poses"].items():
        xyz = P[:, :3, 3]
        tr["pos"][inst] = (tr["G"][:3, :3] @ xyz.T).T + tr["G"][:3, 3]

    t = tr["timestamps"]
    tr["tnorm"] = (t - t[0]) / (t[-1] - t[0])       # duration normalized to [0, 1]

    print(f"{tr['label']:<12s} registration RMSE = {tr['rmse']:.3f} mm")

## 5 · Derived signals and metrics

> **In-use masking (applied to every tracking-derived signal).** Per instrument, per
> frame, we build a validity mask and set all tracking-derived data to **NaN** outside it
> — so masked frames are neither plotted nor averaged:
> - **Bipolar**: in use while its tracking is valid.
> - **Cavitron**: in use while tracked **and** within 100 mm of Bipolar (otherwise treated
>   as not in use).
> - **Scissors**: in use while tracked **and** within 100 mm of Bipolar.
> - Any frame flagged as **missing tracking** (§3) is not in use for that instrument.
>
> This masks velocity, acceleration, jerk, angular speed, the 3D trajectory, the
> inter-instrument distance and angle, and the geometric metrics (path length, working
> volume, straightness, idle fraction). **Force** comes from a separate sensor and is
> *not* masked.

**Signals over time**

- **Force magnitude** `|F| = √(fx² + fy² + fz²)` (invariant to registration).
- **Velocity, acceleration, jerk** — 1st/2nd/3rd time-derivatives of tip position,
  magnitudes reported. Position is lightly Savitzky-Golay smoothed before
  differentiating (numerical differentiation amplifies tracking noise); derivatives
  use the **actual timestamps**, so units are mm/s, mm/s², mm/s³.
- **Inter-instrument distance** `dₐᵦ(t) = ‖pₐ(t) − pᵦ(t)‖` for the pairs meant to be
  used together — **Bipolar–Cavitron** and **Bipolar–Scissors** (Cavitron–Scissors is
  *not* used together and is excluded). When the tips are more than `DIST_MAX = 100 mm`
  apart the instruments aren't interacting, so the distance is set to **NaN** and neither
  plotted nor averaged. Rigid registration preserves distances, so these are unaffected by it.
- **Inter-instrument angle** `θₐᵦ(t)` between the two instruments' **long axes** (see
  below), for the same pairs. Set to NaN on frames where either instrument is untracked.

**Instrument long axis (orientation).** The `*TipToWorldTransform` is a *pivot*
calibration: the tip is the frame origin and the `*TipTo*Transform` translation `d` is
the tip position in the tool-marker frame. The shaft therefore runs along the pivot
direction — the line from the tip back to the marker — **not** along a coordinate axis
of the tip frame (for these tools the closest tip axis is only ~0.83–0.91 aligned). We
take the long axis as `â = normalize(−Rᵀd)` in the tip frame and rotate it to world each
frame with the pose's rotation. Angular quantities (inter-instrument angle, angular
speed) follow from it.

**Scalar metrics per trial / instrument**

- **Path length** `L = Σ‖Δp‖` — total distance the tip travels (mm).
- **Angular speed** — rotational speed of the instrument's frame, deg/s *(suggested)*.
- **Net displacement** `‖p_end − p_start‖` and **straightness** `= net / L ∈ (0, 1]`
  (economy of motion; 1 = perfectly straight). *(suggested)*
- **Working volume** — volume of the tip's axis-aligned bounding box, mm³ *(suggested)*.
- **Idle fraction** — share of time the tip moves slower than `IDLE_SPEED` *(suggested)*.
- **Force impulse** `∫|F| dt` and **peak / mean force**, **mean torque** *(suggested)*.

The *(suggested)* metrics are ones I added as commonly-used surgical-dexterity
indicators — keep or drop them as you like.

In [ ]:
def _smooth(x, win):
    if not win or win < 3 or win >= len(x):
        return x
    if win % 2 == 0:
        win += 1
    try:
        from scipy.signal import savgol_filter
        return savgol_filter(x, win, 3, axis=0)
    except Exception:                     # fallback: centered moving average
        k = np.ones(win) / win
        return np.stack([np.convolve(x[:, j], k, mode="same") for j in range(x.shape[1])], 1)


def kinematics(pos, t, win):
    pos = _smooth(pos, win)
    vel = np.gradient(pos, t, axis=0)
    acc = np.gradient(vel, t, axis=0)
    jrk = np.gradient(acc, t, axis=0)
    return {"velocity": np.linalg.norm(vel, axis=1),
            "acceleration": np.linalg.norm(acc, axis=1),
            "jerk": np.linalg.norm(jrk, axis=1)}


# Instrument pairs that are meant to be used together (Cavitron+Scissors are not).
PAIRS = [("Bipolar", "Cavitron"), ("Bipolar", "Scissors")]
PAIR_KEYS = [f"{a}-{b}" for a, b in PAIRS]
PAIR_COLOR = {"Bipolar-Cavitron": "#7A5195", "Bipolar-Scissors": "#EF5675"}
IDLE_SPEED = 5.0        # mm/s; frames slower than this count as idle/dwell (suggested metric)
DIST_MAX = 100.0        # mm; beyond this the instruments aren't interacting -> distance set NaN

def _path_length(P):
    return float(np.sum(np.linalg.norm(np.diff(P, axis=0), axis=1)))

def _trapz(y, x):                                                    # version-independent ∫y dx
    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))

def long_axis_tip(header, inst):
    "Instrument long (shaft) axis as a unit vector in its calibrated Tip frame."
    key = f"{inst}TipTo{inst}Transform"
    if key not in header:
        return None
    T = np.array([float(x) for x in header[key].split()], float).reshape(4, 4)
    R, d = T[:3, :3], T[:3, 3]        # d = tip position in the tool-marker frame (pivot offset)
    a = -R.T @ d                      # direction tip -> marker, expressed in the Tip frame
    n = np.linalg.norm(a)
    return a / n if n > 0 else None

def angular_speed(R_stack, t):
    "Per-frame rotational speed (deg/s) from the relative rotation between frames."
    dR = np.matmul(R_stack[1:], np.transpose(R_stack[:-1], (0, 2, 1)))
    cos = np.clip((np.trace(dR, axis1=1, axis2=2) - 1) / 2, -1, 1)
    sp = np.degrees(np.arccos(cos)) / np.diff(t)
    return np.concatenate([sp[:1], sp])            # pad to length N

# distance gate: an instrument is "in use" only while within DIST_MAX of Bipolar
GATE_BY = {b: (a, b) for a, b in PAIRS}     # Cavitron -> (Bipolar, Cavitron), Scissors -> (Bipolar, Scissors)

for tr in trials:
    t = tr["timestamps"]
    ts = tr["tracking_status"]
    n = len(t)
    tr["fmag"] = np.linalg.norm(tr["force"][:, :3], axis=1)          # |force| (force sensor, not masked)
    tr["tmag"] = np.linalg.norm(tr["force"][:, 3:], axis=1)          # |torque|
    tr["impulse"] = _trapz(tr["fmag"], t)                            # force impulse ∫|F|dt

    # raw tip-tip distances (before any masking) — drive both the gate and the distance metric
    raw_dist = {f"{a}-{b}": np.linalg.norm(tr["pos"][a] - tr["pos"][b], axis=1)
                for a, b in PAIRS if a in tr["pos"] and b in tr["pos"]}

    # ---- per-instrument "in use" validity mask -------------------------------------
    # Bipolar: valid while tracked. Cavitron/Scissors: tracked AND within DIST_MAX of Bipolar.
    # ALL tracking-derived signals below are set to NaN outside this mask.
    valid = {}
    for inst in tr["pos"]:
        v = (ts[inst] == 1)
        if inst in GATE_BY:
            a, b = GATE_BY[inst]
            key = f"{a}-{b}"
            if key in raw_dist:
                v = v & (ts[a] == 1) & (raw_dist[key] <= DIST_MAX)
        valid[inst] = v
    tr["valid"] = valid
    tr["inuse_frac"] = {inst: float(valid[inst].mean()) for inst in valid}

    # ---- kinematics: computed on the full track, then masked to in-use frames ------
    tr["kin"] = {}
    for inst in tr["pos"]:
        k = kinematics(tr["pos"][inst], t, SMOOTH_WINDOW)
        tr["kin"][inst] = {m: np.where(valid[inst], arr, np.nan) for m, arr in k.items()}

    # ---- long-axis orientation + angular speed (masked) ----------------------------
    tr["axis"], tr["ang_speed"] = {}, {}
    for inst in tr["poses"]:
        a_tip = long_axis_tip(tr["header"], inst)
        Rw = tr["poses"][inst][:, :3, :3]                # tip-frame orientation in world, per frame
        tr["axis"][inst] = (Rw @ a_tip) if a_tip is not None else None
        v = valid.get(inst, np.ones(n, bool))
        tr["ang_speed"][inst] = np.where(v, angular_speed(Rw, t), np.nan)

    # ---- geometric / economy metrics over in-use frames only -----------------------
    tr["pathlen"], tr["netdisp"], tr["straightness"] = {}, {}, {}
    tr["bbox_vol"], tr["idle_frac"] = {}, {}
    for inst, P in tr["pos"].items():
        v = valid[inst]
        idx = np.flatnonzero(v)
        if idx.size >= 2:
            seg_ok = v[1:] & v[:-1]                       # count only steps between two in-use frames
            L = float(np.sum(np.linalg.norm(np.diff(P, axis=0), axis=1)[seg_ok]))
            net = float(np.linalg.norm(P[idx[-1]] - P[idx[0]]))
            bbox = float(np.prod(np.ptp(P[v], axis=0)))
            idle = float(np.mean(tr["kin"][inst]["velocity"][v] < IDLE_SPEED))
        else:
            L = net = bbox = idle = np.nan
        tr["pathlen"][inst] = L
        tr["netdisp"][inst] = net
        tr["straightness"][inst] = (net / L) if (L and L > 0) else np.nan
        tr["bbox_vol"][inst] = bbox
        tr["idle_frac"][inst] = idle

    # ---- inter-instrument distance: NaN where > DIST_MAX or either instrument untracked
    tr["dist"] = {}
    for a, b in PAIRS:
        key = f"{a}-{b}"
        if key in raw_dist:
            d = raw_dist[key].astype(float).copy()
            d[(d > DIST_MAX) | (ts[a] == 0) | (ts[b] == 0)] = np.nan
            tr["dist"][key] = d

    # ---- inter-instrument angle: NaN where either instrument is not in use ----------
    tr["angle"] = {}
    for a, b in PAIRS:
        key = f"{a}-{b}"
        if tr["axis"].get(a) is not None and tr["axis"].get(b) is not None:
            cos = np.clip(np.sum(tr["axis"][a] * tr["axis"][b], axis=1), -1, 1)
            ang = np.degrees(np.arccos(cos))
            ang[(~valid[a]) | (~valid[b])] = np.nan
            tr["angle"][key] = ang

    # ---- masked positions for trajectory plots (in-use frames only) ----------------
    tr["pos_plot"] = {inst: np.where(valid[inst][:, None], P, np.nan)
                      for inst, P in tr["pos"].items()}

KIN_UNITS = {"velocity": "mm/s", "acceleration": "mm/s²", "jerk": "mm/s³"}
# a light->dark ramp to distinguish the trials within one participant
def trial_colors(n):
    return plt.cm.cividis(np.linspace(0.15, 0.85, max(n, 1)))
print("Derived force, kinematics, geometry, inter-instrument distance and orientation metrics.")

## 6 · Force magnitude — per participant

In [ ]:
n = len(participants)
ncol = min(3, n); nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.2 * nrow),
                         squeeze=False, sharex=True)
for ax, (pid, ptrials) in zip(axes.flat, participants.items()):
    cols = trial_colors(len(ptrials))
    for tr, c in zip(ptrials, cols):
        ax.plot(tr["tnorm"], tr["fmag"], lw=1.0, color=c, label=f"T{tr['trial']}")
    ax.set_title(f"Participant {pid}")
    ax.set_xlabel("normalized time"); ax.set_ylabel("|force|  (N)")
    ax.margins(x=0); ax.legend(title="trial", fontsize=8)
for ax in axes.flat[n:]:
    ax.set_visible(False)
fig.suptitle("Force magnitude  √(fx²+fy²+fz²)  per participant", fontweight="bold")
fig.tight_layout()
plt.show()

## 7 · Velocity, acceleration and jerk — one figure per participant

Rows are the three motion metrics; columns are the instruments. Each line is one of
the participant's trials, against normalized time.

In [ ]:
metrics = ["velocity", "acceleration", "jerk"]
for pid, ptrials in participants.items():
    fig, axes = plt.subplots(len(metrics), len(INSTRUMENTS),
                             figsize=(4.6 * len(INSTRUMENTS), 2.9 * len(metrics)),
                             squeeze=False, sharex=True)
    cols = trial_colors(len(ptrials))
    for r, metric in enumerate(metrics):
        for c, inst in enumerate(INSTRUMENTS):
            ax = axes[r][c]
            for tr, col in zip(ptrials, cols):
                if inst in tr["kin"]:
                    ax.plot(tr["tnorm"], tr["kin"][inst][metric], lw=0.9, color=col,
                            label=f"T{tr['trial']}" if (r == 0 and c == 0) else None)
            if r == 0:
                ax.set_title(inst, color=INST_COLOR[inst])
            if c == 0:
                ax.set_ylabel(f"{metric}\n({KIN_UNITS[metric]})")
            if r == len(metrics) - 1:
                ax.set_xlabel("normalized time")
            ax.margins(x=0)
    axes[0][0].legend(title="trial", fontsize=8, loc="upper right")
    fig.suptitle(f"Motion derivatives — participant {pid}", fontweight="bold")
    fig.tight_layout()
    plt.show()

## 8 · 3D instrument trajectories — one figure per participant

Rows are the participant's trials, columns are the instruments. Each registered tip
path is drawn as a 3D line colored from the start (dark) to the end (yellow) of the
recording. Only **in-use** frames are drawn (§5) — the line breaks over frames where
the instrument is untracked or (Cavitron/Scissors) more than 100 mm from Bipolar. All
trials share the common registered frame.

In [ ]:
def _color_line3d(ax, xyz, tnorm, lw=1.6):
    pts = xyz.reshape(-1, 1, 3)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    good = ~np.isnan(segs).any(axis=(1, 2))          # skip segments touching a masked frame
    lc = Line3DCollection(segs[good], cmap=TIME_CMAP, array=tnorm[:-1][good], linewidth=lw)
    lc.set_clim(0, 1)                                # fix color scale to normalized time [0,1]
    ax.add_collection3d(lc)
    return lc

for pid, ptrials in participants.items():
    nrow, ncol = len(ptrials), len(INSTRUMENTS)
    fig = plt.figure(figsize=(5.6 * ncol, 4.6 * nrow))
    lc = None
    for r, tr in enumerate(ptrials):
        for c, inst in enumerate(INSTRUMENTS):
            ax = fig.add_subplot(nrow, ncol, r * ncol + c + 1, projection="3d")
            P = tr["pos_plot"].get(inst)                 # in-use frames only (rest are NaN)
            if P is None or not np.any(~np.isnan(P)):
                ax.set_axis_off(); continue
            lc = _color_line3d(ax, P, tr["tnorm"])
            ax.set_title(f"T{tr['trial']} · {inst}", color=INST_COLOR[inst], fontsize=10)
            ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)"); ax.set_zlabel("z (mm)")
            rng = (np.nanmax(np.nanmax(P, 0) - np.nanmin(P, 0)) / 2) or 1
            mid = np.nanmean(P, axis=0)
            for setlim, m in zip((ax.set_xlim, ax.set_ylim, ax.set_zlim), mid):
                setlim(m - rng, m + rng)
            ax.view_init(elev=20, azim=-60)
    if lc is not None:
        cb = fig.colorbar(lc, ax=fig.axes, shrink=0.5, pad=0.02)
        cb.set_label("normalized time")
    fig.suptitle(f"Registered tip trajectories — participant {pid}", fontweight="bold")
    plt.show()

## 9 · Inter-instrument distance

Tip-to-tip distance over time (**along-trial**, one figure per participant) and its
trial average (**summative**, grouped by participant), for the pairs used together:
**Bipolar–Cavitron** and **Bipolar–Scissors**. Frames where the tips are more than
`DIST_MAX = 100 mm` apart are set to NaN (instruments not interacting) — the line
breaks there and those frames are excluded from the averages. Small distances mean the
instruments are working close together (bimanual coordination / proximity analysis).

In [ ]:
present_pairs = [k for k in PAIR_KEYS if any(k in tr["dist"] for tr in trials)]

# --- along-trial: one figure per participant, one subplot per pair ---
for pid, ptrials in participants.items():
    fig, axes = plt.subplots(1, len(present_pairs),
                             figsize=(5.2 * len(present_pairs), 3.4),
                             squeeze=False, sharex=True)
    cols = trial_colors(len(ptrials))
    for c, pair in enumerate(present_pairs):
        ax = axes[0][c]
        for tr, col in zip(ptrials, cols):
            if pair in tr["dist"]:
                ax.plot(tr["tnorm"], tr["dist"][pair], lw=1.0, color=col, label=f"T{tr['trial']}")
        ax.set_title(pair.replace("-", " – "), color=PAIR_COLOR.get(pair, "#333"))
        ax.set_xlabel("normalized time"); ax.margins(x=0)
        if c == 0:
            ax.set_ylabel("distance (mm)"); ax.legend(title="trial", fontsize=8)
    fig.suptitle(f"Inter-instrument distance — participant {pid}", fontweight="bold")
    fig.tight_layout(); plt.show()

# --- summative: mean distance per pair, grouped by participant (NaN-aware) ---
def _nanmean(a):
    a = np.asarray(a, float)
    return float(np.nanmean(a)) if np.any(~np.isnan(a)) else np.nan

def _pair_grouped_bar(value_of, ylabel, title):
    pids = list(participants)
    fig, ax = plt.subplots(figsize=(1.8 * len(pids) + 4, 4.2))
    x = np.arange(len(pids)); k = max(len(present_pairs), 1); w = 0.8 / k
    for i, pair in enumerate(present_pairs):
        means, errs = [], []
        for pid in pids:
            vals = [value_of(tr, pair) for tr in participants[pid] if pair in tr["dist"]]
            vals = [v for v in vals if v is not None and np.isfinite(v)]
            means.append(np.mean(vals) if vals else 0.0)
            errs.append(np.std(vals) if len(vals) > 1 else 0.0)
        means, errs = np.asarray(means), np.asarray(errs)
        ax.bar(x + (i - (k - 1) / 2) * w, means, width=w * 0.95,
               yerr=np.vstack([np.minimum(errs, means), errs]),
               label=pair.replace("-", " – "), color=PAIR_COLOR.get(pair, None),
               error_kw=dict(lw=1, capsize=3, ecolor="#555"))
    ax.set_xticks(x); ax.set_xticklabels(pids); ax.set_xlabel("participant")
    ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=8); ax.margins(y=0.15)
    fig.tight_layout(); plt.show()

_pair_grouped_bar(lambda tr, pair: _nanmean(tr["dist"][pair]),
                  "mean distance (mm)", "Average inter-instrument distance (< 100 mm only)")

## 10 · Instrument orientation

Angle between the two instruments' **long axes** over time (**along-trial**) and its
trial average (**summative**), for **Bipolar–Cavitron** and **Bipolar–Scissors**. The
long axis is the pivot (shaft) direction recovered in §5; 0° means the two shafts are
parallel, 180° anti-parallel. Frames where either instrument is untracked are NaN.

In [ ]:
present_ang = [k for k in PAIR_KEYS if any(k in tr["angle"] for tr in trials)]

# --- along-trial angle: one figure per participant, one subplot per pair ---
for pid, ptrials in participants.items():
    fig, axes = plt.subplots(1, len(present_ang),
                             figsize=(5.2 * len(present_ang), 3.4),
                             squeeze=False, sharex=True)
    cols = trial_colors(len(ptrials))
    for c, pair in enumerate(present_ang):
        ax = axes[0][c]
        for tr, col in zip(ptrials, cols):
            if pair in tr["angle"]:
                ax.plot(tr["tnorm"], tr["angle"][pair], lw=1.0, color=col, label=f"T{tr['trial']}")
        ax.set_title(pair.replace("-", " – "), color=PAIR_COLOR.get(pair, "#333"))
        ax.set_xlabel("normalized time"); ax.set_ylim(0, 180); ax.margins(x=0)
        if c == 0:
            ax.set_ylabel("angle (deg)"); ax.legend(title="trial", fontsize=8)
    fig.suptitle(f"Inter-instrument angle — participant {pid}", fontweight="bold")
    fig.tight_layout(); plt.show()

# --- summative: mean angle per pair, grouped by participant ---
def _nanmean_a(a):
    a = np.asarray(a, float)
    return float(np.nanmean(a)) if np.any(~np.isnan(a)) else np.nan

pids = list(participants)
fig, ax = plt.subplots(figsize=(1.8 * len(pids) + 4, 4.2))
x = np.arange(len(pids)); k = max(len(present_ang), 1); w = 0.8 / k
for i, pair in enumerate(present_ang):
    means, errs = [], []
    for pid in pids:
        vals = [_nanmean_a(tr["angle"][pair]) for tr in participants[pid] if pair in tr["angle"]]
        vals = [v for v in vals if v is not None and np.isfinite(v)]
        means.append(np.mean(vals) if vals else 0.0)
        errs.append(np.std(vals) if len(vals) > 1 else 0.0)
    means, errs = np.asarray(means), np.asarray(errs)
    ax.bar(x + (i - (k - 1) / 2) * w, means, width=w * 0.95,
           yerr=np.vstack([np.minimum(errs, means), errs]),
           label=pair.replace("-", " – "), color=PAIR_COLOR.get(pair, None),
           error_kw=dict(lw=1, capsize=3, ecolor="#555"))
ax.set_xticks(x); ax.set_xticklabels(pids); ax.set_xlabel("participant")
ax.set_ylabel("mean angle (deg)"); ax.set_title("Average inter-instrument angle")
ax.legend(fontsize=8); ax.margins(y=0.15)
fig.tight_layout(); plt.show()

## 11 · Summative figures

Cross-participant comparison as grouped bars (x = participant). **Force** is
summarized per participant (single sensor); **velocity / acceleration / jerk / path
length / straightness / % tracked / angular speed** are per instrument. **% tracked**
is the share of frames with valid (non-frozen) tracking. Error bars are the standard
deviation across each participant's trials (clipped at zero, since the quantities are
non-negative).

In [ ]:
def _grouped_bar(ax, participant_ids, series, ylabel, title, colors=None):
    "series: dict label -> (means[np], errs[np]); one group of bars per participant."
    x = np.arange(len(participant_ids))
    k = len(series)
    w = 0.8 / k
    for i, (lab, (means, errs)) in enumerate(series.items()):
        means, errs = np.asarray(means, float), np.asarray(errs, float)
        yerr = np.vstack([np.minimum(errs, means), errs])
        off = (i - (k - 1) / 2) * w
        ax.bar(x + off, means, width=w * 0.95, yerr=yerr, label=lab,
               color=None if colors is None else colors[i],
               error_kw=dict(lw=1, capsize=3, ecolor="#555"))
    ax.set_xticks(x); ax.set_xticklabels(participant_ids)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.set_xlabel("participant")
    ax.margins(y=0.15)
    if k > 1:
        ax.legend(fontsize=8)

pids = list(participants)

# mean/std of any per-trial scalar, per instrument, across each participant's trials
def inst_group_stats(value_fn):
    means, errs = {}, {}
    for inst in INSTRUMENTS:
        m, e = [], []
        for pid in pids:
            vals = [value_fn(tr, inst) for tr in participants[pid] if inst in tr["pos"]]
            vals = [v for v in vals if v is not None and np.isfinite(v)]
            m.append(np.mean(vals) if vals else 0.0)
            e.append(np.std(vals) if len(vals) > 1 else 0.0)
        means[inst], errs[inst] = m, e
    return means, errs

fig, axes = plt.subplots(4, 2, figsize=(13, 17))

# (a) average force magnitude per participant (single sensor)
fmeans = [np.mean([tr["fmag"].mean() for tr in participants[pid]]) for pid in pids]
ferrs = [np.std([tr["fmag"].mean() for tr in participants[pid]]) if len(participants[pid]) > 1
         else 0.0 for pid in pids]
_grouped_bar(axes[0][0], pids, {"|force|": (fmeans, ferrs)},
             "|force|  (N)", "Average force magnitude per participant", colors=["#0072B2"])

# (b–g) per-instrument metrics
panels = [
    (axes[0][1], "velocity",     KIN_UNITS["velocity"],     lambda tr, i: np.nanmean(tr["kin"][i]["velocity"])),
    (axes[1][0], "acceleration", KIN_UNITS["acceleration"], lambda tr, i: np.nanmean(tr["kin"][i]["acceleration"])),
    (axes[1][1], "jerk",         KIN_UNITS["jerk"],         lambda tr, i: np.nanmean(tr["kin"][i]["jerk"])),
    (axes[2][0], "path length",  "mm",                      lambda tr, i: tr["pathlen"][i]),
    (axes[2][1], "straightness", "net / path",              lambda tr, i: tr["straightness"][i]),
    (axes[3][0], "% tracked",    "% of frames",             lambda tr, i: 100 * tr["tracking_status"][i].mean()),
    (axes[3][1], "angular speed", "deg/s",                  lambda tr, i: np.nanmean(tr["ang_speed"][i])),
]
for ax, name, unit, fn in panels:
    means, errs = inst_group_stats(fn)
    series = {inst: (means[inst], errs[inst]) for inst in INSTRUMENTS}
    _grouped_bar(ax, pids, series, unit, f"Average {name} per instrument",
                 colors=[INST_COLOR[i] for i in INSTRUMENTS])
axes[3][0].set_ylim(0, 100)                       # % tracked is a percentage

fig.suptitle("Cross-participant comparison", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

## 12 · Per-trial statistics table

One row per trial with all summary metrics, plus a per-participant aggregate (mean
across each participant's trials). Both tables are written to CSV next to the zip so
they can be opened in a spreadsheet or fed into statistical tests.

All per-instrument and pair columns reflect the **in-use masking** (frames where an
instrument is untracked, or a gated instrument is > 100 mm from Bipolar, are excluded).
Columns include, per instrument: **% tracked**, **% in use**, **path length** (mm),
mean/peak **speed**, mean **acceleration** and **jerk**, **straightness**, **working
volume** (mm³), **idle fraction** and mean **angular speed** (deg/s); per pair:
mean/min/max **inter-instrument distance** (mm) and mean **angle** (deg); plus per-trial
**force** mean/peak, **impulse** and mean **torque** (force is not masked).

In [ ]:
import pandas as pd

def _nanmean(a):
    a = np.asarray(a, float)
    return float(np.nanmean(a)) if np.any(~np.isnan(a)) else np.nan

def _nanmax(a):
    a = np.asarray(a, float)
    return float(np.nanmax(a)) if np.any(~np.isnan(a)) else np.nan

def _r(x, nd):
    x = float(x)
    return round(x, nd) if np.isfinite(x) else np.nan

rows = []
for tr in trials:
    row = {
        "participant": tr["participant"],
        "trial": tr["trial"],
        "file": tr["name"],
        "n_frames": len(tr["timestamps"]),
        "duration_s": round(float(tr["timestamps"][-1] - tr["timestamps"][0]), 3),
        "reg_rmse_mm": round(float(tr["rmse"]), 4),
        "force_mean_N": round(float(tr["fmag"].mean()), 5),
        "force_peak_N": round(float(tr["fmag"].max()), 5),
        "force_impulse": round(float(tr["impulse"]), 4),
        "torque_mean": round(float(tr["tmag"].mean()), 5),
    }
    for inst in INSTRUMENTS:
        if inst not in tr["pos"]:
            continue
        k = tr["kin"][inst]
        row[f"{inst}_tracked_pct"] = round(100 * float(tr["tracking_status"][inst].mean()), 2)
        row[f"{inst}_inuse_pct"] = round(100 * float(tr["inuse_frac"][inst]), 2)
        row[f"{inst}_path_mm"] = _r(tr["pathlen"][inst], 1)
        row[f"{inst}_speed_mean"] = _r(_nanmean(k["velocity"]), 2)
        row[f"{inst}_speed_peak"] = _r(_nanmax(k["velocity"]), 2)
        row[f"{inst}_accel_mean"] = _r(_nanmean(k["acceleration"]), 2)
        row[f"{inst}_jerk_mean"] = _r(_nanmean(k["jerk"]), 1)
        row[f"{inst}_straightness"] = _r(tr["straightness"][inst], 3)
        row[f"{inst}_workvol_mm3"] = _r(tr["bbox_vol"][inst], 0)
        row[f"{inst}_idle_frac"] = _r(tr["idle_frac"][inst], 3)
        row[f"{inst}_angspeed_dps"] = _r(_nanmean(tr["ang_speed"][inst]), 2)
    for pair in PAIR_KEYS:
        if pair in tr["dist"]:                          # distance ignores >100 mm gaps (NaN)
            d = tr["dist"][pair]
            valid = np.any(~np.isnan(d))
            row[f"dist_{pair}_mean"] = round(float(np.nanmean(d)), 1) if valid else np.nan
            row[f"dist_{pair}_min"] = round(float(np.nanmin(d)), 1) if valid else np.nan
            row[f"dist_{pair}_max"] = round(float(np.nanmax(d)), 1) if valid else np.nan
        if pair in tr["angle"]:
            a = tr["angle"][pair]
            row[f"angle_{pair}_mean"] = (round(float(np.nanmean(a)), 1)
                                         if np.any(~np.isnan(a)) else np.nan)
    rows.append(row)

per_trial = pd.DataFrame(rows)

# per-participant aggregate: mean of the numeric per-trial metrics
num_cols = [c for c in per_trial.columns if c not in ("participant", "trial", "file", "n_frames")]
per_participant = (per_trial.groupby("participant")[num_cols]
                   .mean().round(3).reset_index())
per_participant.insert(1, "n_trials",
                       per_trial.groupby("participant").size().values)

# save next to the zip
OUT_DIR = os.path.dirname(os.path.abspath(DATA_ZIP))
out_trial = os.path.join(OUT_DIR, "metrics_per_trial.csv")
out_part = os.path.join(OUT_DIR, "metrics_per_participant.csv")
per_trial.to_csv(out_trial, index=False)
per_participant.to_csv(out_part, index=False)
print("wrote", out_trial, "and", out_part)

from IPython.display import display
print("\nPer-trial statistics:")
display(per_trial)
print("Per-participant aggregate (mean over trials):")
display(per_participant)

---
### Notes & assumptions

- **Data layout.** One zip (`DATA_ZIP`) containing `<participant>/<trial>.igs.mha`
  subfolders; it is extracted into `WORK_DIR` on every run. Participants are the
  folders that directly contain the `.igs.mha` files (a wrapper folder in the zip is
  ignored); trials are numbered by filename order within each participant folder.
- **Single reference frame.** All trials of all participants are registered onto
  `trials[0]`'s fiducials, so everything is comparable. Change which trial is the
  anchor by reordering, or set `ref_pts` yourself in §4.
- **Fiducial order matters.** Registration assumes `BipolarCollectedPoint0..3`
  correspond across trials; a high RMSE in §4 usually means a swapped/mis-collected point.
- **Missing tracking & in-use masking.** A frame is flagged missing when an instrument's
  4×4 pose is bit-for-bit identical to the previous frame (`tr["tracking_status"][inst]`:
  0=missing, 1=tracked). The §5 in-use mask (`tr["valid"][inst]`) combines that flag with
  the 100 mm distance gate for Cavitron/Scissors, and **every tracking-derived signal is
  set to NaN outside it** — velocities are still differentiated across the full track
  first, then masked, so gaps don't corrupt the derivatives. `tr["inuse_frac"][inst]`
  reports the share of frames an instrument was in use. A genuinely motionless instrument
  would also read as missing, but real tracking always jitters, so exact equality only
  occurs when the pose is frozen/held.
- **Smoothing.** `SMOOTH_WINDOW` sets the Savitzky-Golay window used before
  differentiation. Larger = smoother but more lag; `0` differentiates the raw track.
- **Force is a single sensor**, not per-instrument, so it is summarized per
  participant/trial. Torque magnitude is also parsed (`tr["tmag"]`) if you want it.
- **Inter-instrument distance** is computed only for pairs used together
  (Bipolar–Cavitron, Bipolar–Scissors; Cavitron–Scissors is excluded). Distances above
  `DIST_MAX = 100 mm` are set to NaN and dropped from plots and averages — tune the
  threshold to your task. It is computed on registered positions, but a rigid transform
  preserves distance, so registration does not change it.
- **Instrument long axis / orientation.** The pivot-calibrated `*TipToWorldTransform`
  puts the tip at the frame origin; the shaft runs along the pivot direction
  `â = normalize(−Rᵀd)` (from the constant `*TipTo*Transform`), **not** along a tip-frame
  coordinate axis. The inter-instrument **angle** is between these axes (0°=parallel,
  180°=anti-parallel), NaN where either instrument is untracked. **Angular speed** is
  the rotational speed of the instrument frame (deg/s). If your tools had a full spin
  calibration you could instead use a fixed tip-frame axis — swap `long_axis_tip`.
- **Suggested metrics.** `straightness = net displacement / path length` is a simple
  economy-of-motion measure — note it drops toward 0 for paths that loop back near
  their start even when efficient. `working volume` is the axis-aligned bounding-box
  volume (swap for a convex-hull volume via `scipy.spatial.ConvexHull` if you prefer).
  `idle fraction` uses the `IDLE_SPEED` threshold (mm/s) — tune it to your setup.
  Other metrics worth adding if useful: **submovement count** (velocity peaks, a
  smoothness proxy), **spectral arc length / dimensionless jerk** (established movement-
  smoothness scores), and **speed correlation** between the two instruments (bimanual
  coordination).